# Env

In [ ]:
import os
import re
import glob
import json
from tqdm import tqdm

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn.functional as F
from torch.distributions import Distribution, Categorical

from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          AutoModelWithLMHead,
                          AutoModelForTokenClassification,
                          GenerationConfig,
                          Trainer,
                          TrainingArguments)
from datasets import load_dataset

In [ ]:
# work dir
work_dir = '/home/ubuntu/nlp-practice'

In [ ]:
%cd {work_dir}
!pwd

In [ ]:
# tokeinzer warning disable
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# GPT를 이용한 문장분류 (NSMC)

## Train

In [ ]:
# !python train_gpt_nsmc.py

## Test

In [ ]:
!ls results/gpt-nsmc

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("./results/gpt-nsmc/checkpoint-9376")
tokenizer.pad_token = "<pad>"

model = AutoModelForSequenceClassification.from_pretrained("./results/gpt-nsmc/checkpoint-9376")

In [ ]:
model.eval()
device = next(model.parameters()).device

idx2label = {0: "부정", 1: "긍정"}

In [ ]:
dataset = load_dataset("e9t/nsmc")

In [ ]:
test_data = dataset["test"].select(np.random.randint(0, 10000, 10))

for row in test_data:
    document = row["document"]
    x = tokenizer(
        document,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    ).to(device)
    
    logit = model(**x).logits[0]
    prob = F.softmax(logit, dim=-1)
    # |prob| = (batch_size, output_dim)

    y = prob.argmax(dim=-1)
    # |y| = (batch_size,)

    print(f"{idx2label[y.item()]}\t{prob[y].item():.4f}\t{document}")

## Infer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("./results/gpt-nsmc/checkpoint-9376")
tokenizer.pad_token = "<pad>"

model = AutoModelForSequenceClassification.from_pretrained("./results/gpt-nsmc/checkpoint-9376")

In [ ]:
model.eval()
device = next(model.parameters()).device

idx2label = {0: "부정", 1: "긍정"}

In [ ]:
while True:
    print("input> ", end="")
    line = str(input())
    if len(line) == 0:
        break

    x = tokenizer(
        line,
        truncation=True,
        max_length=512,
        return_tensors="pt",
    ).to(device)

    logit = model(**x).logits[0]
    prob = F.softmax(logit, dim=-1)
    # |prob| = (batch_size, output_dim)

    y = prob.argmax(dim=-1)
    # |y| = (batch_size,)

    print(f"{idx2label[y.item()]}\t{prob[y].item():.4f}\t{line}")

# GPT를 이용한 문장생성

In [ ]:
device = torch.device("cpu")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("skt/kogpt2-base-v2")
tokenizer.pad_token = "<pad>"

model = AutoModelWithLMHead.from_pretrained("skt/kogpt2-base-v2")

In [ ]:
line = "근육이 커지기 위해서는"
# 입력을 token_id로 변환
x = tokenizer(
        line,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )["input_ids"].to(device)

with torch.no_grad():
    for i in range(128):
        # print(x)
        logits = model(x).logits
        prob = F.softmax(logits[:, -1, :], dim=-1)  # 마지막 출력의 예측 값 (다음 단어)
        dist = Categorical(prob)
        token_id = dist.sample()                    # sampling
        if token_id == tokenizer.eos_token_id:
            break
        token_id = token_id.reshape(-1, 1)          # |1| -> \1, 1\
        x = torch.cat((x, token_id), dim=-1)        # 다음 단어를 입력으로 추가
        
text = tokenizer.decode(x.reshape(-1).cpu().numpy())
print(text)

In [ ]:
generation_config = GenerationConfig(
        max_new_tokens=128,   # 생성할 수 있는 최대 새 토큰 수
        early_stopping=True,  # 생성 알고리즘 조기 종료를 활성화
        do_sample=True,       # 다음 토큰을 생성할 때 때 확률 분포에서 무작위로 샘플 여부
        num_beams=8,          # 빔 서치(beam search)에서 사용할 빔 수
        use_cache=True,       # 키 캐시(Key-Value Cache)를 사용할지 여부
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        decoder_start_token_id=tokenizer.bos_token_id,
        repetition_penalty=1.2,
        length_penalty=1.0,
    )

In [ ]:
# 근육이 커지기 위해서는
while True:
    print("input> ", end="")
    line = str(input())
    if len(line) == 0:
        break

    x = tokenizer(
        line,
        truncation=True,
        max_length=128,
        return_tensors="pt",
    )["input_ids"].to(device)

    beam_output = model.generate(
        input_ids=x,
        generation_config=generation_config,
    )
    result = tokenizer.decode(beam_output[0], skip_special_tokens=True)

    print(f"- {result}\n")

# BERT를 이용한 토큰분류 (NER)

## 전처리

In [ ]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained("klue/roberta-base")
tokenizer

In [ ]:
# read ner files
fn_list = glob.glob("./data/NER/*_NER.txt")
fn_list = sorted(fn_list)
len(fn_list)

In [ ]:
def get_tags(fn):
    tags = []
    with open(fn) as f:
        for line in f:
            line = line.strip()
            if line.startswith('## '):
                pass
            else:
                tokens = line.split()
                if len(tokens) == 4:
                    tags.append(tokens[3])
    return tags

In [ ]:
# tag list 확인
all_tags = []
for fn in tqdm(fn_list):
    all_tags.extend(get_tags(fn))
all_tags = list(dict.fromkeys(all_tags))
all_tags

In [ ]:
# tag에 일련번호 부여
label2idx = {"[PAD]": 0, "O": 1}
for tag in all_tags:
    if tag not in label2idx:
        label2idx[tag] = len(label2idx)
label2idx

In [ ]:
with open("./data/NER/label2idx.json", "w") as f:
    json.dump(label2idx, f, indent=4)

In [ ]:
# data에서 text와 label 조회
def get_datas(fn):
    texts, labels = [], []
    with open(fn) as f:
        sharp_count = 0
        text, label = '', ''
        for line in f:
            line = line.strip()
            if line.startswith('## '):
                sharp_count += 1
                if sharp_count == 1:
                    pass
                elif sharp_count == 2:
                    text = ' '.join(line[3:].strip().split())
                elif sharp_count == 3:
                    label = ' '.join(line[3:].strip().split())
                    texts.append(text)
                    labels.append(label)
            else:
                sharp_count = 0
                text, label = "", ""
    return texts, labels

In [ ]:
all_texts, all_labels = [], []
for fn in tqdm(fn_list):
    texts, labels = get_datas(fn)
    all_texts.extend(texts)
    all_labels.extend(labels)
len(all_texts), len(all_labels)

In [ ]:
data = {'text': all_texts, 'label_text': all_labels}
df = pd.DataFrame(data)
df

In [ ]:
PATTERN = re.compile("<([^<]+):([A-Z]{3})>")

def find_tags(text, label_text):
    tag_list = []

    matchs = PATTERN.finditer(label_text)
    n_match = 0
    for match in matchs:
        w = match[1]
        t = match[2]
        s = match.start()
        e = match.end()
        i1 = s - 6 * n_match
        i2 = e - 6 * n_match - 6 - 1
        assert w == text[i1:i2 + 1]
        tag_list.append((w, t, i1, i2))
        n_match += 1

    return tag_list

In [ ]:
row = df.iloc[3]
print(row.text)
print(row.label_text)

In [ ]:
# 윈문 기준 tag 조회
tag_list = find_tags(row.text, row.label_text)
tag_list

In [ ]:
def _is_whitespace(c):
    if c == " " or c == "\t" or c == "\r" or c == "\n" or ord(c) == 0x202F:
        return True
    return False

In [ ]:
def _tokenize_whitespace(text):
    word_tokens = []
    char_to_word = []
    prev_is_whitespace = True

    for c in text:
        if _is_whitespace(c):
            prev_is_whitespace = True
        else:
            if prev_is_whitespace:
                word_tokens.append(c)
            else:
                word_tokens[-1] += c
            prev_is_whitespace = False
        char_to_word.append(len(word_tokens) - 1)

    return word_tokens, char_to_word

In [ ]:
# whitespace 단위로 tokenize
word_tokens, char_to_word = _tokenize_whitespace(row.text)
print(word_tokens)
print(char_to_word)

In [ ]:
for i, w in enumerate(word_tokens):
    print(w, end="\t[ ")
    for idx in char_to_word:
        if idx == i:
            print(idx, end=" ")
    print("]")

In [ ]:
def _tokenize_vocab(tokenizer, word_tokens):
    word_to_sub = []
    sub_tokens = []
    for (i, word) in enumerate(word_tokens):
        word_to_sub.append(len(sub_tokens))
        tokens = tokenizer.tokenize(word)
        for token in tokens:
            sub_tokens.append(token)
    return sub_tokens, word_to_sub

In [ ]:
# whitespace 단위 -> sub word 단위 (by tokenizer)
sub_tokens, word_to_sub = _tokenize_vocab(tokenizer, word_tokens)
print(sub_tokens)
print(word_to_sub)

In [ ]:
# 데이터 확인
for i in range(len(word_to_sub) - 1):
    print(sub_tokens[word_to_sub[i]:word_to_sub[i+1]])

In [ ]:
def _improve_span(tokenizer, sub_tokens, token_start, token_end, char_answer):
    token_answer = " ".join(tokenizer.tokenize(char_answer))
    for new_start in range(token_start, token_end + 1):
        for new_end in range(token_end, new_start - 1, -1):
            text_span = " ".join(sub_tokens[new_start : (new_end + 1)])
            if not token_answer.startswith("##") and text_span.startswith("##"):
                text_span = text_span[2:]
            if text_span == token_answer:
                return (new_start, new_end)
    return (token_start, token_end)

In [ ]:
# 각 token 별 정답
sub_labels = ['O'] * len(sub_tokens)
print(sub_labels)

In [ ]:
# token 위치에 tag 값 할당
for w, t, i1, i2 in tag_list:
    word_start = char_to_word[i1]
    word_end = char_to_word[i2]
    token_start = word_to_sub[word_start]
    if word_end < len(word_to_sub) - 1:
        token_end = word_to_sub[word_end + 1] - 1
    else:
        token_end = len(sub_tokens) - 1
    token_start, token_end = _improve_span(tokenizer, sub_tokens, token_start, token_end, w)
    print(w, t, sub_tokens[token_start:token_end+1])

    b_tag = True
    for j in range(token_start, token_end + 1):
        sub_labels[j] = f'B-{t}' if b_tag else f'I-{t}'
        b_tag = False

In [ ]:
# 결과확인
for w, t in zip(sub_tokens, sub_labels):
    print(w, ":", t)

In [ ]:
dataset = []
# 전체 데이터 처리 및 저장
for i, row in tqdm(df.iterrows(), total=len(df)):
    # 윈문 기준 tag 조회
    tag_list = find_tags(row.text, row.label_text)
    # whitespace 단위로 tokenize
    word_tokens, char_to_word = _tokenize_whitespace(row.text)
    # whitespace 단위 -> sub word 단위 (by tokenizer)
    sub_tokens, word_to_sub = _tokenize_vocab(tokenizer, word_tokens)
    # 각 token 별 정답
    sub_labels = ['O'] * len(sub_tokens)
    # token 위치에 tag 값 할당
    for w, t, i1, i2 in tag_list:
        word_start = char_to_word[i1]
        word_end = char_to_word[i2]
        token_start = word_to_sub[word_start]
        if word_end < len(word_to_sub) - 1:
            token_end = word_to_sub[word_end + 1] - 1
        else:
            token_end = len(sub_tokens) - 1
        token_start, token_end = _improve_span(tokenizer, sub_tokens, token_start, token_end, w)

        b_tag = True
        for j in range(token_start, token_end + 1):
            sub_labels[j] = f'B-{t}' if b_tag else f'I-{t}'
            b_tag = False
    dataset.append({
        "text": row.text,
        "label_text": row.label_text,
        "tokens": [tokenizer.cls_token] + sub_tokens[:510] + [tokenizer.sep_token],
        "labels":  ['O'] + sub_labels[:510] + ['O']
    })

In [ ]:
train_dataset, valid_dataset = train_test_split(
    dataset,
    test_size=0.2,
    random_state=1234
)
len(train_dataset), len(valid_dataset)

In [ ]:
with open("./data/NER/train_dataset.jsonl", "w") as f:
    for row in train_dataset:
        f.write(json.dumps(row, ensure_ascii=False))
        f.write("\n")

In [ ]:
with open("./data/NER/valid_dataset.jsonl", "w") as f:
    for row in valid_dataset:
        f.write(json.dumps(row, ensure_ascii=False))
        f.write("\n")

## Train

In [ ]:
# !python train_bert_ner.py

## Test

In [ ]:
!ls results/bert-ner

In [ ]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained("klue/roberta-base")
tokenizer

In [ ]:
# model
model = AutoModelForTokenClassification.from_pretrained(
    "./results/bert-ner/checkpoint-1446"
)

In [ ]:
from train_bert_ner import NERDataset
valid_dataset = NERDataset("./data/NER/valid_dataset.jsonl", tokenizer, label2idx)

In [ ]:
model.eval()
device = next(model.parameters()).device

idx2label = {v:k for k, v in label2idx.items()}
idx2label

In [ ]:
test_index = np.random.randint(0, 3800, 10)

for idx in test_index:
    input_ids, token_type_ids, attention_mask, _ = valid_dataset[idx]
    x = {
        "input_ids": input_ids.unsqueeze(0).to(device),
        "token_type_ids": token_type_ids.unsqueeze(0).to(device),
        "attention_mask": attention_mask.unsqueeze(0).to(device),
    }
    
    logit = model(**x).logits[0][1:-1]  # drop [CLS], [SEP]
    prob = F.softmax(logit, dim=-1)
    # |prob| = (seq_len, output_dim)

    tokens = tokenizer.convert_ids_to_tokens(input_ids[1:-1])
    y_preds = prob.argmax(dim=-1)
    # |y| = (seq_len,)

    print("****", tokenizer.decode(input_ids[1:-1]))
    for t, i in zip(tokens, y_preds):
        if i != 1:
            print(t, ":", idx2label[i.item()])

## Infer

In [ ]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained("klue/roberta-base")
tokenizer

In [ ]:
# model
model = AutoModelForTokenClassification.from_pretrained(
    "./results/bert-ner/checkpoint-1446"
)

In [ ]:
model.eval()
device = next(model.parameters()).device

idx2label = {v:k for k, v in label2idx.items()}
idx2label

In [ ]:
while True:
    line = str(input("input> "))
    if len(line) == 0:
        break

    x = tokenizer(
        line,
        truncation=True,
        max_length=256,
        return_tensors="pt",
    ).to(device)

    logit = model(**x).logits[0][1:-1]  # drop [CLS], [SEP]
    prob = F.softmax(logit, dim=-1)
    # |prob| = (seq_len, output_dim)

    tokens = tokenizer.tokenize(line)
    y_preds = prob.argmax(dim=-1)
    # |y| = (seq_len)
    
    print("****", line)
    for t, i in zip(tokens, y_preds):
        if i != 1:
            print(t, ":", idx2label[i.item()])